# Active Learning for Frame Weights - Sphere Hypothesis Class

This notebook extends the adaptive active learning algorithm to support **negative weights**
by sampling from the unit sphere instead of the simplex.

**Key differences from simplex version:**
- Hypothesis class: unit sphere $\|w\|_2 = 1$ (weights can be negative)
- Intensity: $r(\omega) = \sum_{j \in \text{active}} |w_j| \cdot |\Delta_j|$
- Preference: $\delta(\omega) = \sum_{j \in \text{active}} w_j \cdot \Delta_j$ (signed)
- Resampling: perturb + project back to sphere

In [ ]:
# Imports
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Set, Optional
from dataclasses import dataclass
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
import seaborn as sns

sns.set_style("whitegrid")
np.random.seed(42)

# Import shared data structures and query generation from core
from core import (
    Patient, PairwiseQuery, QueryResponse,
    FEATURE_NAMES, FEATURE_RANGES,
    compute_diameter, compute_stable_convergence_metrics,
    generate_random_patient, compute_frame_gaps,
    generate_query_activating_frames, generate_candidate_queries,
    compute_frame_uncertainties, generate_adaptive_candidates,
    check_epsilon_pareto,
    extract_transcript, fit_bradley_terry,
)

In [ ]:
# Configuration
TAU = 1.0           # Intensity threshold
TAU_PRIME = 0.2     # Resolvability threshold
LAMBDA_X = 1.0      # Query scaling factor

print("Configuration:")
print(f"  Features: {FEATURE_NAMES}")
print(f"  Thresholds: tau={TAU}, tau'={TAU_PRIME}")
print(f"  lambda(X) = {LAMBDA_X}")
print(f"  Hypothesis class: unit sphere (weights can be negative)")
print(f"  Intensity: r(w) = sum |w_j| * |gap_j|")

## Sphere Sampling

Instead of sampling from the simplex (non-negative, sum-to-1), we sample uniformly
from the unit sphere $\{w \in \mathbb{R}^d : \|w\|_2 = 1\}$.

This allows negative weights, meaning a feature can have a *negative* contribution
to the decision.

In [ ]:
def sample_from_sphere(n_samples: int, dim: int,
                       random_state: Optional[int] = None) -> np.ndarray:
    """
    Sample uniformly from the unit sphere ||w||_2 = 1.
    
    Method: sample from standard normal, then normalize each vector.
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    z = np.random.randn(n_samples, dim)
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    samples = z / np.maximum(norms, 1e-10)
    return samples


def resample_from_feasible_set_sphere(existing_samples: np.ndarray,
                                      target_n_samples: int,
                                      noise_scale: float = 0.05) -> np.ndarray:
    """
    Replenish feasible set by perturbing existing samples and
    re-projecting onto the unit sphere.
    """
    n_existing, dim = existing_samples.shape
    indices = np.random.choice(n_existing, size=target_n_samples, replace=True)
    new_samples = existing_samples[indices].copy()
    
    noise = np.random.randn(target_n_samples, dim) * noise_scale
    new_samples = new_samples + noise
    
    # Project back to sphere
    norms = np.linalg.norm(new_samples, axis=1, keepdims=True)
    new_samples = new_samples / np.maximum(norms, 1e-10)
    
    return new_samples


# Test
samples = sample_from_sphere(10, 5, random_state=42)
print("Sphere samples (first 3):")
print(samples[:3])
print(f"Norms: {np.linalg.norm(samples, axis=1)[:3]}")
print(f"Min value: {samples.min():.3f}, Max value: {samples.max():.3f}")
print()
resampled = resample_from_feasible_set_sphere(samples, target_n_samples=10)
print("Resampled norms:", np.linalg.norm(resampled, axis=1)[:3])

## Core Computations with |w_i| * |gap_i| Intensity

The key change: intensity uses absolute weights.

$$r(\omega) = \sum_{j \in \text{active}} |w_j| \cdot |\Delta_j|$$

$$\delta(\omega) = \sum_{j \in \text{active}} w_j \cdot \Delta_j$$

This means a weight of $-0.3$ contributes the same intensity as $+0.3$,
but pulls the preference in the opposite direction.

In [ ]:
def compute_aggregate_scores_sphere(gaps: np.ndarray,
                                     weights: np.ndarray,
                                     active_frames: Set[int]) -> Tuple[float, float]:
    """
    Compute aggregate scores with |w_i| * |gap_i| intensity.
    
    delta(w) = sum_{j in active} w_j * gap_j       (signed preference)
    r(w)     = sum_{j in active} |w_j| * |gap_j|   (intensity, always >= 0)
    """
    if len(active_frames) == 0:
        return 0.0, 0.0
    
    active_list = sorted(list(active_frames))
    active_gaps = gaps[active_list]
    active_weights = weights[active_list]
    
    delta_omega = np.dot(active_weights, active_gaps)
    r_omega = np.dot(np.abs(active_weights), np.abs(active_gaps))
    
    return delta_omega, r_omega


def predict_response_sphere(query: PairwiseQuery,
                            weights: np.ndarray,
                            tau: float = TAU,
                            lambda_x: float = LAMBDA_X,
                            tau_prime: float = TAU_PRIME) -> str:
    """
    Predict response using |w_i| * |gap_i| intensity.
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    delta_omega, r_omega = compute_aggregate_scores_sphere(gaps, weights, active_frames)
    
    if r_omega < tau:
        return 'indifferent'
    if r_omega >= tau and np.abs(delta_omega) < tau_prime * r_omega:
        return 'incomparable'
    elif r_omega >= tau and delta_omega >= tau_prime * r_omega:
        return 'left'
    elif r_omega >= tau and delta_omega <= -tau_prime * r_omega:
        return 'right'
    else:
        return 'indifferent'


# Test
gaps = np.array([2.0, -5.0, -2.0, 10.0, -2.0])
active = {0, 1, 3, 4}
w_pos = np.array([0.2, 0.3, 0.0, 0.4, 0.1])
w_neg = np.array([0.2, -0.3, 0.0, 0.4, -0.1])

d1, r1 = compute_aggregate_scores_sphere(gaps, w_pos, active)
d2, r2 = compute_aggregate_scores_sphere(gaps, w_neg, active)

print(f"Positive weights {w_pos}:")
print(f"  delta={d1:.3f}, r={r1:.3f}")
print(f"Negative weights {w_neg}:")
print(f"  delta={d2:.3f}, r={r2:.3f}")
print(f"\nNote: r uses |w_i|*|gap_i|, so r is the same for both")

In [ ]:
def filter_samples_by_response_sphere(samples: np.ndarray,
                                       query: PairwiseQuery,
                                       observed_response: str,
                                       tau: float = TAU,
                                       lambda_x: float = LAMBDA_X,
                                       tau_prime: float = TAU_PRIME) -> np.ndarray:
    """
    Filter samples using |w_i| * |gap_i| intensity.
    """
    gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
    
    if len(active_frames) == 0:
        if observed_response == 'indifferent':
            return samples
        else:
            return np.empty((0, samples.shape[1]))
    
    active_list = sorted(list(active_frames))
    active_gaps = gaps[active_list]
    
    active_weights = samples[:, active_list]
    delta_omegas = np.dot(active_weights, active_gaps)
    r_omegas = np.dot(np.abs(active_weights), np.abs(active_gaps))
    
    predicted = np.empty(len(samples), dtype=object)
    
    for idx in range(len(samples)):
        r_omega = r_omegas[idx]
        delta_omega = delta_omegas[idx]
        
        if r_omega < tau:
            predicted[idx] = 'indifferent'
        elif r_omega >= tau and np.abs(delta_omega) < tau_prime * r_omega:
            predicted[idx] = 'incomparable'
        elif r_omega >= tau and delta_omega >= tau_prime * r_omega:
            predicted[idx] = 'left'
        elif r_omega >= tau and delta_omega <= -tau_prime * r_omega:
            predicted[idx] = 'right'
        else:
            predicted[idx] = 'indifferent'
    
    consistent_mask = (predicted == observed_response)
    return samples[consistent_mask]


# Test
samples = sample_from_sphere(1000, 5, random_state=42)
p1 = Patient(elderlyDep=2, lifeYearsGained=20, obesity=1, weeklyWorkhours=40, yearsWaiting=5)
p2 = Patient(elderlyDep=0, lifeYearsGained=15, obesity=3, weeklyWorkhours=30, yearsWaiting=3)
query = PairwiseQuery(p1, p2)

print(f"Initial samples: {len(samples)}")
for resp in ['left', 'right', 'indifferent', 'incomparable']:
    filtered = filter_samples_by_response_sphere(samples, query, resp)
    print(f"  Consistent with '{resp}': {len(filtered)} ({100*len(filtered)/len(samples):.1f}%)")

In [ ]:
def select_query_by_uncertainty_sphere(candidates: List[PairwiseQuery],
                                        samples: np.ndarray,
                                        tau: float = TAU,
                                        lambda_x: float = LAMBDA_X,
                                        tau_prime: float = TAU_PRIME) -> Tuple[PairwiseQuery, Dict[str, float]]:
    """
    Select query maximizing uncertainty, using |w_i|*|gap_i| intensity.
    """
    if len(candidates) == 0:
        raise ValueError("No candidate queries provided")
    if len(samples) == 0:
        return candidates[0], {'uncertainty': 0.0}
    
    best_query = None
    best_uncertainty = -1
    best_info = {}
    
    for query in candidates:
        gaps, active_frames = compute_frame_gaps(query, lambda_x, tau)
        
        if len(active_frames) == 0:
            continue
        
        active_list = sorted(list(active_frames))
        active_gaps = gaps[active_list]
        active_weights = samples[:, active_list]
        
        delta_omegas = np.dot(active_weights, active_gaps)
        r_omegas = np.dot(np.abs(active_weights), np.abs(active_gaps))
        
        response_counts = {'left': 0, 'right': 0, 'indifferent': 0, 'incomparable': 0}
        
        for idx in range(len(samples)):
            r_omega = r_omegas[idx]
            delta_omega = delta_omegas[idx]
            
            if r_omega < tau:
                response_counts['indifferent'] += 1
            elif r_omega >= tau and np.abs(delta_omega) < tau_prime * r_omega:
                response_counts['incomparable'] += 1
            elif r_omega >= tau and delta_omega >= tau_prime * r_omega:
                response_counts['left'] += 1
            elif r_omega >= tau and delta_omega <= -tau_prime * r_omega:
                response_counts['right'] += 1
            else:
                response_counts['indifferent'] += 1
        
        total = len(samples)
        probs = [count / total for count in response_counts.values() if count > 0]
        
        if len(probs) <= 1:
            uncertainty = 0.0
        else:
            uncertainty = -sum(p * np.log2(p) for p in probs)
        
        if uncertainty > best_uncertainty:
            best_uncertainty = uncertainty
            best_query = query
            best_info = {
                'uncertainty': uncertainty,
                'response_counts': response_counts.copy(),
                'active_frames': active_frames
            }
    
    if best_query is None:
        best_query = candidates[0]
        best_info = {'uncertainty': 0.0}
    
    return best_query, best_info

## Adaptive Learning Loop (Sphere)

In [ ]:
def active_learning_loop_adaptive_sphere(
    n_initial_samples: int = 20000,
    epsilon_pareto: float = 0.15,
    max_iterations: int = 100,
    n_candidates: int = 50,
    top_k_uncertain: int = 3,
    min_samples: int = 100,
    resample_threshold: int = 0,
    oracle_weights: Optional[np.ndarray] = None,
    verbose: bool = True
) -> Tuple[np.ndarray, List[Dict]]:
    """
    Adaptive active learning on the unit sphere with |w_i|*|gap_i| intensity.
    """
    n_features = len(FEATURE_NAMES)
    samples = sample_from_sphere(n_initial_samples, n_features, random_state=42)
    history = []
    
    if verbose:
        print(f"ADAPTIVE Active Learning (Sphere Hypothesis Class)")
        print(f"{'='*60}")
        print(f"Initial samples: {len(samples)}")
        print(f"Intensity: r(w) = sum |w_j| * |gap_j|")
        print(f"Stopping criterion: epsilon-Pareto with epsilon = {epsilon_pareto}")
        print(f"Max iterations: {max_iterations}")
        print(f"Adaptive targeting: Top-{top_k_uncertain} uncertain frames\n")
    
    for iteration in range(max_iterations):
        is_pareto, max_distance = check_epsilon_pareto(samples, epsilon_pareto)
        
        if verbose:
            print(f"Iteration {iteration + 1}")
            print(f"  Feasible samples: {len(samples)}")
            print(f"  Max L1 distance: {max_distance:.4f}")
            print(f"  epsilon-Pareto (epsilon={epsilon_pareto}): {'YES' if is_pareto else 'NO'}")
        
        if is_pareto:
            if verbose:
                print(f"\nConverged! All rules are {epsilon_pareto}-Pareto optimal")
                print(f"   (Max L1 distance {max_distance:.4f} <= {epsilon_pareto})")
            break
        
        uncertainties = compute_frame_uncertainties(samples)
        if verbose:
            top_uncertain_indices = np.argsort(uncertainties)[::-1][:top_k_uncertain]
            top_uncertain_names = [FEATURE_NAMES[i] for i in top_uncertain_indices]
            print(f"  Most uncertain frames: {top_uncertain_names}")
        
        candidates = generate_adaptive_candidates(
            samples,
            n_candidates=n_candidates,
            top_k_uncertain=top_k_uncertain
        )
        
        if len(candidates) == 0:
            if verbose:
                print("  Warning: No valid candidates generated")
            break
        
        query, query_info = select_query_by_uncertainty_sphere(candidates, samples)
        
        if verbose:
            print(f"  Query uncertainty: {query_info.get('uncertainty', 0):.3f} bits")
            print(f"  Active frames: {sorted(query_info.get('active_frames', set()))}")
        
        if oracle_weights is not None:
            response = predict_response_sphere(query, oracle_weights)
        else:
            print(f"\n{query}")
            print("Please respond: left/right/indifferent/incomparable")
            response = input("Response: ").strip().lower()
        
        if verbose:
            print(f"  Response: {response}")
        
        samples_before = len(samples)
        samples = filter_samples_by_response_sphere(samples, query, response)
        samples_after = len(samples)
        
        if verbose:
            print(f"  Filtered: {samples_before} -> {samples_after} samples")
        
        if samples_after < resample_threshold and samples_after > 0:
            samples = resample_from_feasible_set_sphere(samples, min_samples)
            if verbose:
                print(f"  Resampled to {len(samples)} samples")
        elif samples_after == 0:
            if verbose:
                print("  ERROR: No consistent samples remain!")
            break
        
        history.append({
            'iteration': iteration + 1,
            'query': query,
            'response': response,
            'max_distance': max_distance,
            'is_epsilon_pareto': is_pareto,
            'n_samples': samples_after,
            'uncertainty': query_info.get('uncertainty', 0),
            'active_frames': query_info.get('active_frames', set()),
            'frame_uncertainties': uncertainties.copy()
        })
        
        if verbose:
            print()
    
    learned_weights = samples.mean(axis=0)
    
    if verbose:
        print(f"\nLearned weights:")
        for i, (name, weight) in enumerate(zip(FEATURE_NAMES, learned_weights)):
            print(f"  {name:20s}: {weight:.4f}")
        
        if oracle_weights is not None:
            print(f"\nGround truth weights:")
            for i, (name, weight) in enumerate(zip(FEATURE_NAMES, oracle_weights)):
                print(f"  {name:20s}: {weight:.4f}")
            
            l1_error = np.abs(learned_weights - oracle_weights).sum()
            print(f"\nL1 error: {l1_error:.4f}")
    
    return learned_weights, history

## Run Experiment

Test with ground truth weights that include negative values.

In [ ]:
np.random.seed(42)

# Ground truth weights on the sphere (including negative weights)
true_weights = np.array([0.5, 0.4, -0.3, 0.2, -0.1])
true_weights = true_weights / np.linalg.norm(true_weights)  # Normalize to unit sphere

print("True weights (unit sphere):")
for name, w in zip(FEATURE_NAMES, true_weights):
    print(f"  {name:20s}: {w:.4f}")
print(f"  ||w||_2 = {np.linalg.norm(true_weights):.4f}")

In [ ]:
learned_weights, history = active_learning_loop_adaptive_sphere(
    n_initial_samples=100000,
    epsilon_pareto=0.15,
    max_iterations=100,
    n_candidates=50,
    top_k_uncertain=3,
    resample_threshold=0,
    oracle_weights=true_weights,
    verbose=True
)

## Visualization

In [ ]:
def plot_sphere_learning_progress(history: List[Dict],
                                   learned_weights: np.ndarray,
                                   true_weights: Optional[np.ndarray] = None,
                                   feature_names: List[str] = FEATURE_NAMES):
    """
    Visualization for sphere-based active learning.
    """
    if not history:
        print("No history to plot")
        return
    
    iterations = [h['iteration'] for h in history]
    diameters = [h.get('diameter', h.get('max_distance', 0)) for h in history]
    n_samples_list = [h['n_samples'] for h in history]
    uncertainties = [h['uncertainty'] for h in history]
    n_active_frames = [len(h.get('active_frames', set())) for h in history]
    
    n_features = len(feature_names)
    frame_activation = np.zeros((len(history), n_features))
    for i, h in enumerate(history):
        active = h.get('active_frames', set())
        for frame_idx in active:
            if frame_idx < n_features:
                frame_activation[i, frame_idx] = 1
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Sphere Active Learning Progress', fontsize=16, fontweight='bold')
    
    # Plot 1: Diameter convergence
    ax = axes[0, 0]
    ax.plot(iterations, diameters, 'o-', linewidth=2, markersize=6, color='steelblue')
    if len(diameters) > 3:
        from scipy.ndimage import uniform_filter1d
        window_size = min(5, len(diameters))
        smoothed = uniform_filter1d(diameters, size=window_size)
        ax.plot(iterations, smoothed, '--', linewidth=2.5, color='darkblue', alpha=0.7, label='Trend')
        ax.legend()
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Max Distance', fontsize=11)
    ax.set_title('Feasible Set Max Distance', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Active frames
    ax = axes[0, 1]
    ax.plot(iterations, n_active_frames, 'o-', linewidth=2, markersize=6, color='darkviolet')
    ax.axhline(y=n_features, color='gray', linestyle='--', alpha=0.5, label=f'Max={n_features}')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Number of Active Frames', fontsize=11)
    ax.set_title('Active Frame Count', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_ylim(0, n_features + 0.5)
    
    # Plot 3: Frame activation heatmap
    ax = axes[0, 2]
    im = ax.imshow(frame_activation.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Feature Index', fontsize=11)
    ax.set_title('Frame Activation Pattern', fontsize=12, fontweight='bold')
    ax.set_yticks(range(n_features))
    ax.set_yticklabels([f'{i}' for i in range(n_features)])
    plt.colorbar(im, ax=ax, label='Active (1) / Inactive (0)')
    
    # Plot 4: Uncertainty
    ax = axes[1, 0]
    ax.plot(iterations, uncertainties, 'o-', linewidth=2, markersize=6, color='mediumseagreen')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Uncertainty (bits)', fontsize=11)
    ax.set_title('Query Uncertainty', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 5: Sample count
    ax = axes[1, 1]
    ax.plot(iterations, n_samples_list, 'o-', linewidth=2, markersize=6, color='coral')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Number of Samples', fontsize=11)
    ax.set_title('Feasible Sample Count', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 6: Weight comparison
    ax = axes[1, 2]
    x_pos = np.arange(len(feature_names))
    width = 0.35
    
    bars1 = ax.bar(x_pos - width/2, learned_weights, width, label='Learned', color='steelblue', alpha=0.8)
    if true_weights is not None:
        bars2 = ax.bar(x_pos + width/2, true_weights, width, label='True', color='coral', alpha=0.8)
    
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Feature', fontsize=11)
    ax.set_ylabel('Weight', fontsize=11)
    ax.set_title('Learned vs True Weights', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n" + "="*60)
    print("SPHERE ACTIVE LEARNING STATISTICS")
    print("="*60)
    print(f"Total iterations: {len(history)}")
    print(f"Average active frames per query: {np.mean(n_active_frames):.2f}")
    print(f"Final max distance: {diameters[-1]:.4f}")
    if history[-1].get('is_epsilon_pareto') is not None:
        print(f"Epsilon-Pareto optimal: {history[-1]['is_epsilon_pareto']}")
    print(f"Final samples: {n_samples_list[-1]}")
    print(f"Average uncertainty: {np.mean(uncertainties):.3f} bits")
    
    print("\nFrame activation frequency:")
    activation_counts = frame_activation.sum(axis=0)
    for i, count in enumerate(activation_counts):
        pct = 100 * count / len(history)
        print(f"  Feature {i} ({feature_names[i]}): {int(count)}/{len(history)} queries ({pct:.1f}%)")
    
    if true_weights is not None:
        l1_error = np.abs(learned_weights - true_weights).sum()
        l2_error = np.linalg.norm(learned_weights - true_weights)
        cos_sim = np.dot(learned_weights, true_weights) / (
            np.linalg.norm(learned_weights) * np.linalg.norm(true_weights) + 1e-10)
        print(f"\nL1 error: {l1_error:.4f}")
        print(f"L2 error: {l2_error:.4f}")
        print(f"Cosine similarity: {cos_sim:.4f}")
    print("="*60)


plot_sphere_learning_progress(history, learned_weights, true_weights)

## Experiment: All-Positive Weights (Compare to Simplex)

Run the sphere model with all-positive weights to compare convergence behavior
with the simplex version.

In [ ]:
np.random.seed(42)

# All-positive weights on the sphere (comparable to simplex case)
true_weights_pos = np.array([0.1, 0.5, 0.3, 0.1, 0.0])
true_weights_pos = true_weights_pos / np.linalg.norm(true_weights_pos)  # project to sphere

print("True weights (all positive, on sphere):")
for name, w in zip(FEATURE_NAMES, true_weights_pos):
    print(f"  {name:20s}: {w:.4f}")

learned_pos, history_pos = active_learning_loop_adaptive_sphere(
    n_initial_samples=100000,
    epsilon_pareto=0.15,
    max_iterations=100,
    n_candidates=50,
    top_k_uncertain=3,
    resample_threshold=0,
    oracle_weights=true_weights_pos,
    verbose=True
)

plot_sphere_learning_progress(history_pos, learned_pos, true_weights_pos)